### Stations

In [2]:
import requests
import pandas as pd
import time

In [32]:

url = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"
params = {"size": 5000, "code_bss" :"09937X0135/P23B, 09934X0087/P18B, 10193X0151/P29B, 09938X0164/P12B, 10192X0095/P21B"}
headers = {"accept": "application/json"}

response = requests.get(url, params=params, headers=headers)
response.raise_for_status() 

data = response.json()
df_station = pd.DataFrame(data["data"])
df_station = df_station.rename(columns={"code_bss": "code_station", "y": "longitude", "x": "latitude"})

print(data.get("count"), "résultats au total")
display(df_station.describe())
display(df_station.head())

display(df_station["nom_commune"].value_counts())

5 résultats au total


,latitude,longitude,nb_mesures_piezo,profondeur_investigation
count,5.000000,5.000000,5.000000,5.000000
mean,4.913446,43.581604,8082.800000,15.902000
std,0.082374,0.042741,148.056408,4.747159
min,4.811397,43.539905,7900.000000,12.000000
25%,4.871875,43.550764,7963.000000,13.010000
50%,4.897662,43.578046,8114.000000,15.000000
75%,4.961161,43.590416,8186.000000,15.500000
max,5.025133,43.648887,8251.000000,24.000000


,code_station,urn_bss,date_debut_mesure,date_fin_mesure,code_commune_insee,nom_commune,latitude,longitude,codes_bdlisa,urns_bdlisa,...,altitude_station,nb_mesures_piezo,code_departement,nom_departement,libelle_pe,profondeur_investigation,codes_masse_eau_edl,noms_masse_eau_edl,urns_masse_eau_edl,date_maj
0,10192X0095/P21B,http://services.ades.eaufrance.fr/pointeau/101...,2002-01-23,2026-07-15,13004,Arles,4.811397,43.539905,[561AF00],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,4.0,7900,13,Bouches-du-Rhône,Arles - Négreiron,12.00,[DG104],[Cailloutis de la Crau],[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Thu Jul 31 21:19:00 CEST 2025
1,09938X0164/P12B,http://services.ades.eaufrance.fr/pointeau/099...,2003-01-30,2026-07-15,13063,Miramas,5.025133,43.590416,[561AE00],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,60.0,8186,13,Bouches-du-Rhône,Miramas - les Cabasses,24.00,[DG513],[Formations variées du bassin versant de la To...,[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Fri Jun 28 07:31:38 CEST 2024
2,09937X0135/P23B,http://services.ades.eaufrance.fr/pointeau/099...,2003-06-02,2026-07-15,13097,Saint-Martin-de-Crau,4.871875,43.578046,[561AF00],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,24.0,8114,13,Bouches-du-Rhône,Saint-Martin-de-Crau - le Petit Carton,15.00,[DG104],[Cailloutis de la Crau],[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Fri Jun 28 07:31:38 CEST 2024
3,09934X0087/P18B,http://services.ades.eaufrance.fr/pointeau/099...,2003-01-02,2026-07-15,13097,Saint-Martin-de-Crau,4.961161,43.648887,[561AF00],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,59.0,8251,13,Bouches-du-Rhône,Saint-Martin-de-Crau - Mas Archimbaud,15.50,[DG104],[Cailloutis de la Crau],[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Wed Apr 23 23:16:06 CEST 2025
4,10193X0151/P29B,http://services.ades.eaufrance.fr/pointeau/101...,2002-01-23,2026-07-15,13047,Istres,4.897662,43.550764,[561AF00],[http://reseau.eaufrance.fr/geotraitements/bdl...,...,24.0,7963,13,Bouches-du-Rhône,Istres -Peyre Estève,13.01,[DG104],[Cailloutis de la Crau],[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Fri Jun 28 07:31:38 CEST 2024


nom_commune
Saint-Martin-de-Crau    2
Arles                   1
Miramas                 1
Istres                  1
Name: count, dtype: int64

### Piezometre

In [ ]:
url = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques"
headers = {"accept": "application/json"}

def fetch_next_page(first_url: str,
               code_station: str) -> pd.DataFrame:
    data_page = []
    next_url = first_url
    params = {
        "code_bss": code_station,
        "page": 1,
        "size": 5000
    }

    while next_url:
        try:
            r = requests.get(next_url, params=params, headers=headers, timeout=(5, 30))
            r.raise_for_status()
            print (r.url)

        except requests.exceptions.Timeout:
            print("Timeout, nouvelle tentative...")
            time.sleep(2)
            continue 

        data = r.json()
        data_page.extend(data["data"])
        next_url = data.get("next")
        params = None

        time.sleep(1)
        
    df = pd.DataFrame(data_page)
    return df





df_p= []

for i, (code_station, lat, lon) in enumerate(zip(df_station["code_station"], df_station["latitude"], df_station["longitude"])):
    if i>0 :
        break

    print (f"----- Station: {code_station}, latitude:{lat}-longitude:{lon}")
    df_cache = fetch_next_page(url, code_station)
    df_p.append(df_cache)


piezometre_df = pd.concat(df_p, ignore_index=True)

print(len(piezometre_df), "résultats au total")

date_debut= piezometre_df['date_mesure'].min()
date_fin= piezometre_df['date_mesure'].max()

print(f"Début de la mesure {date_debut}, fin de la mesure {date_fin}")
print("\n")
display(piezometre_df)

----- Station: 08081X0026/SE.20, latitude:44.923232442-longitude:1.148944586
https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques?code_bss=08081X0026%2FSE.20&page=1&size=5000
https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques?code_bss=08081X0026/SE.20&page=2&size=5000
7960 résultats au total
Début de la mesure 1996-09-05, fin de la mesure 2026-07-15




,code_bss,bss_id,urn_bss,date_mesure,timestamp_mesure,niveau_nappe_eau,mode_obtention,statut,qualification,code_continuite,nom_continuite,code_producteur,nom_producteur,code_nature_mesure,nom_nature_mesure,profondeur_nappe
0,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,1996-09-05,841881600000,152.52,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,1.40
1,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,1998-05-14,895104000000,152.93,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,0.99
2,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,1999-08-17,934848000000,152.93,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,0.99
3,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2000-08-24,967075200000,152.98,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,0.94
4,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2001-10-17,1003276800000,152.78,Valeur mesurée,Donnée contrôlée niveau 1,Correcte,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),0,Inconnue,1.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7955,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2026-07-11,1783728000000,152.69,Valeur mesurée,Donnée brute,Non qualifié,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),N,Naturel,1.23
7956,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2026-07-12,1783814400000,152.64,Valeur mesurée,Donnée brute,Non qualifié,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),N,Naturel,1.28
7957,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2026-07-13,1783900800000,152.64,Valeur mesurée,Donnée brute,Non qualifié,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),N,Naturel,1.28
7958,08081X0026/SE.20,BSS001YSAR,http://services.ades.eaufrance.fr/pointeau/080...,2026-07-14,1783990800000,152.64,Valeur mesurée,Donnée brute,Non qualifié,2,Point lié au point précédent,327,Service Géologique Régional d'Aquitaine (327),N,Naturel,1.28
